In [5]:
from openai import Omit, OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

query = "answer the question: Informace ohledně kácení dřevin a kdo je oznamovatel, return only the sparlq query, nothing else"

vector_store = client.vector_stores.create(
    name="knowledge_base",
    chunking_strategy=Omit()
)
print(vector_store.id)

result = client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    chunking_strategy=Omit(),
    file_id="file-LqTBHUNQznBk18NEbQB68Q"
)
print(result)

result = client.vector_stores.files.list(
    vector_store_id=vector_store.id
)
print(result)

from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [f"{vector_store.id}"]
    }]
)
print(response)
full_text = "\n".join(
    content.text 
    for msg in response.output 
    for content in msg.content 
    if hasattr(content, 'text')
)
print(full_text)


vs_690f26b11f688191b920bf1af7755fec
VectorStoreFile(id='file-LqTBHUNQznBk18NEbQB68Q', created_at=1762600626, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_690f26b11f688191b920bf1af7755fec', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))
SyncCursorPage[VectorStoreFile](data=[], has_more=False, object='list', first_id=None, last_id=None)
Response(id='resp_09281ae3397f205b00690f26b412c8819ea1cf45c981553b79', created_at=1762600628.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_09281ae3397f205b00690f26b544d8819eadfbe1717c0bc333', content=[ResponseOutputText(annotations=[], text='SELECT ?info ?oznamovatel\nWHERE {\n  ?kaceni rdf:type :KaceniDrevin .\n  ?kaceni :informace ?info .\n  ?kaceni :oznamovatel ?ozna

In [20]:
from pyld import jsonld
import json
import requests
from urllib.parse import urlparse

class CachingDocumentLoader:
    """Document loader that caches contexts and prevents recursive loading"""
    
    def __init__(self):
        self.cache = {}
        self.loading = set()  # Track contexts currently being loaded
    
    def __call__(self, url, options={}):
        """Load a document, with caching and recursion prevention"""
        
        # If we're already loading this URL, return empty to break recursion
        if url in self.loading:
            print(f"⚠ Breaking recursive load of: {url}")
            return {
                'contextUrl': None,
                'documentUrl': url,
                'document': {}
            }
        
        # Check cache first
        if url in self.cache:
            print(f"✓ Using cached context: {url}")
            return self.cache[url]
        
        # Mark as currently loading
        self.loading.add(url)
        
        try:
            print(f"→ Fetching context: {url}")
            
            # Fetch the context
            response = requests.get(url, timeout=10, headers={
                'Accept': 'application/ld+json, application/json'
            })
            response.raise_for_status()
            
            # Parse the JSON
            doc = response.json()
            
            # Create the document structure
            result = {
                'contextUrl': None,
                'documentUrl': url,
                'document': doc
            }
            
            # Cache it
            self.cache[url] = result
            print(f"✓ Cached context: {url}")
            
            return result
            
        except Exception as e:
            print(f"✗ Failed to load {url}: {e}")
            # Return empty document on failure
            return {
                'contextUrl': None,
                'documentUrl': url,
                'document': {}
            }
        
        finally:
            # Remove from loading set
            self.loading.discard(url)

# Create the custom loader
loader = CachingDocumentLoader()

# Load your JSON-LD file
with open("./udalosti.jsonld") as f:
    doc = json.load(f)

print("=" * 60)
print("Expanding JSON-LD document...")
print("=" * 60)

try:
    # Expand with custom document loader
    expanded = jsonld.expand(doc, options={
        "documentLoader": loader,
        "base": "https://www.namestnosl.cz/"
    })
    
    print("\n" + "=" * 60)
    print("✓ SUCCESS! Document expanded")
    print("=" * 60)
    
    # Save expanded form
    with open("./udalosti_expanded.jsonld", "w", encoding="utf-8") as f:
        json.dump(expanded, f, indent=2, ensure_ascii=False)
    
    print(f"\nSaved to: udalosti_expanded.jsonld")
    print(f"Size: {len(json.dumps(expanded))} characters")
    print(f"\nFirst item preview:")
    print(json.dumps(expanded[0] if expanded else {}, indent=2)[:500])
    
    # Now parse with rdflib
    print("\n" + "=" * 60)
    print("Loading into RDF graph...")
    print("=" * 60)
    
    import rdflib
    g = rdflib.Graph()
    g.parse(data=json.dumps(expanded), format="json-ld")
    
    print(f"✓ Loaded {len(g)} triples")
    
    # Show some sample triples
    print("\nSample triples:")
    for i, (s, p, o) in enumerate(g):
        if i < 5:
            print(f"  {s}")
            print(f"    → {p}")
            print(f"    → {o}\n")
        else:
            break
    
    # Save as N-Triples for easier inspection
    nt_output = g.serialize(format="nt")
    with open("./opendata-uredni-deska (2).jsonld", "w", encoding="utf-8") as f:
        f.write(nt_output)
    print(f"✓ Saved triples to: udalosti.nt")
    
except Exception as e:
    print("\n" + "=" * 60)
    print(f"✗ ERROR: {e}")
    print("=" * 60)
    
    import traceback
    traceback.print_exc()
    
    # If it still fails, show what contexts were loaded
    print("\n--- Loaded contexts ---")
    for url in loader.cache.keys():
        print(f"  • {url}")

Expanding JSON-LD document...
→ Fetching context: https://ofn.gov.cz/události/2020-07-01/kontexty/událost.jsonld
✓ Cached context: https://ofn.gov.cz/události/2020-07-01/kontexty/událost.jsonld

✗ ERROR: string indices must be integers, not 'str'

--- Loaded contexts ---
  • https://ofn.gov.cz/události/2020-07-01/kontexty/událost.jsonld


Traceback (most recent call last):
  File "/tmp/ipykernel_12327/1554888596.py", line 84, in <module>
    expanded = jsonld.expand(doc, options={
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonld.py", line 163, in expand
    return JsonLdProcessor().expand(input_, options)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonld.py", line 870, in expand
    expanded = self._expand(active_ctx, None, document, options,
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonld.py", line 2233, in _expand
    e = self._expand(
        ^^^^^^^^^^^^^
  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonl

In [9]:
import json
import rdflib
from pyld import jsonld

# 1. Load the raw JSON-LD data
with open("./opendata-uredni-deska-brand.jsonld", 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Use PyLD to flatten the document (resolves context first)
# You need to provide a base IRI for external context resolution to work correctly.
# Use an empty dictionary for the context to ensure all IRIs are expanded.
flattened_data = jsonld.flatten(data, {})

# 3. Dump the result to a string (PyLD output is a Python list/dict)
flattened_string = json.dumps(flattened_data)

# 4. Parse the flattened JSON-LD string using rdflib
# rdflib.Graph().parse() can read from a string using `data` argument
g = rdflib.Graph()
# The document is now flattened, so there are no recursive context imports during parsing.
g.parse(data=flattened_string, format="json-ld")

print(f"Graph parsed successfully with {len(g)} triples.")

JsonLdError: ('Could not expand input before flattening.',)
Type: jsonld.FlattenError
Cause: string indices must be integers, not 'str'  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonld.py", line 912, in flatten
    expanded = self.expand(input_, options)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonld.py", line 870, in expand
    expanded = self._expand(active_ctx, None, document, options,
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonld.py", line 2302, in _expand
    active_ctx = self._process_context(
                 ^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Lamosst/FEL/diplomka/data-catalog-search/data_catalog_env/lib/python3.12/site-packages/pyld/jsonld.py", line 3053, in _process_conte